# Valsalva Maneuver Simulation

**Clinical Application:** Autonomic function testing and cardiac reserve assessment

**Learning Objectives:**
1. Understand the 4 phases of the Valsalva maneuver
2. Simulate hemodynamic responses to forced expiration
3. Interpret autonomic compensatory mechanisms
4. Identify pathological Valsalva responses

**Clinical Relevance:**
- Autonomic neuropathy screening (Freeman 2006)
- Heart failure assessment (NYHA classification)
- Syncope evaluation
- Cardiovascular fitness testing

**Physiological Background:**

The Valsalva maneuver involves forced expiration against a closed glottis (typically 40 mmHg for 10-15 seconds), producing characteristic hemodynamic changes:

- **Phase I:** Onset of strain → ↑ Intrathoracic pressure → Transient ↑ BP
- **Phase II:** Continued strain → ↓ Venous return → ↓ CO → ↓ BP → Baroreflex activation → ↑ HR
- **Phase III:** Release → Rapid ↓ intrathoracic pressure → Further ↓ BP
- **Phase IV:** Recovery → Restored venous return → BP overshoot → Baroreflex → ↓ HR

**Reference:** Goldberger et al. (2019) American Heart Association Scientific Statement

In [ ]:
import sys
sys.path.append('..')
import numpy as np
import matplotlib.pyplot as plt
from src.autonomic.autonomic_nervous_system import AutonomicNervousSystem, AutonomicParameters
from src.validation.benchmarks import PhysiologicalBenchmarks

%matplotlib inline
plt.rcParams['figure.figsize'] = (14, 10)
print("✓ Imports successful")

## Part 1: Normal Valsalva Response

Simulate a healthy individual performing the Valsalva maneuver with standard parameters.

In [ ]:
# Create normal autonomic system
ans = AutonomicNervousSystem()

# Simulate standard Valsalva maneuver
results = ans.simulate_valsalva_maneuver(
    duration=20.0,           # seconds
    strain_duration=10.0,    # seconds
    strain_pressure=40.0,    # mmHg
    dt=0.01
)

# Extract time series
times = np.array(results['time'])
pressure = np.array(results['pressure'])
heart_rate = np.array(results['heart_rate'])
vagal = np.array(results['vagal_tone'])
sympathetic = np.array(results['sympathetic_tone'])
phase = np.array(results['phase'])

# Create comprehensive visualization
fig, axes = plt.subplots(5, 1, figsize=(14, 14), sharex=True)

# Phase annotations
phase_colors = ['lightblue', 'lightyellow', 'lightcoral', 'lightgreen']
phase_labels = ['I: Onset', 'II: Strain', 'III: Release', 'IV: Recovery']

for ax in axes:
    for i in range(1, 5):
        phase_mask = phase == i
        if np.any(phase_mask):
            t_start = times[phase_mask][0]
            t_end = times[phase_mask][-1]
            ax.axvspan(t_start, t_end, alpha=0.2, color=phase_colors[i-1])

# Plot 1: Arterial Pressure
axes[0].plot(times, pressure, 'b-', linewidth=2.5)
axes[0].axhline(93, color='gray', linestyle='--', alpha=0.5, label='Baseline MAP')
axes[0].set_ylabel('Mean Arterial\nPressure (mmHg)', fontsize=11, fontweight='bold')
axes[0].set_title('Valsalva Maneuver - Normal Response', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

# Plot 2: Heart Rate
axes[1].plot(times, heart_rate, 'r-', linewidth=2.5)
axes[1].axhline(75, color='gray', linestyle='--', alpha=0.5, label='Resting HR')
axes[1].set_ylabel('Heart Rate\n(bpm)', fontsize=11, fontweight='bold')
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

# Plot 3: Vagal Tone
axes[2].plot(times, vagal, 'g-', linewidth=2.5, label='Vagal (Parasympathetic)')
axes[2].set_ylabel('Vagal Tone\n(0-1)', fontsize=11, fontweight='bold')
axes[2].set_ylim([0, 1.0])
axes[2].legend(fontsize=10)
axes[2].grid(True, alpha=0.3)

# Plot 4: Sympathetic Tone
axes[3].plot(times, sympathetic, 'orange', linewidth=2.5, label='Sympathetic')
axes[3].set_ylabel('Sympathetic Tone\n(0-1)', fontsize=11, fontweight='bold')
axes[3].set_ylim([0, 1.0])
axes[3].legend(fontsize=10)
axes[3].grid(True, alpha=0.3)

# Plot 5: Phase Diagram
for i in range(1, 5):
    phase_mask = phase == i
    if np.any(phase_mask):
        t_mid = np.mean(times[phase_mask])
        axes[4].text(t_mid, 0.5, phase_labels[i-1], 
                    ha='center', va='center', fontsize=11, fontweight='bold',
                    bbox=dict(boxstyle='round', facecolor=phase_colors[i-1], alpha=0.8))
axes[4].set_xlim([times[0], times[-1]])
axes[4].set_ylim([0, 1])
axes[4].set_xlabel('Time (seconds)', fontsize=12, fontweight='bold')
axes[4].set_ylabel('Phase', fontsize=11, fontweight='bold')
axes[4].set_yticks([])
axes[4].grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

print("\n" + "="*60)
print("NORMAL VALSALVA RESPONSE - KEY METRICS")
print("="*60)

# Compute Valsalva ratio (Phase II max HR / Phase IV min HR)
phase2_mask = phase == 2
phase4_mask = phase == 4

if np.any(phase2_mask) and np.any(phase4_mask):
    hr_phase2_max = np.max(heart_rate[phase2_mask])
    hr_phase4_min = np.min(heart_rate[phase4_mask])
    valsalva_ratio = hr_phase2_max / hr_phase4_min
    
    print(f"\nPhase II (Strain):")
    print(f"  Maximum HR: {hr_phase2_max:.1f} bpm")
    print(f"  Minimum BP: {np.min(pressure[phase2_mask]):.1f} mmHg")
    
    print(f"\nPhase IV (Recovery):")
    print(f"  Minimum HR: {hr_phase4_min:.1f} bpm")
    print(f"  Maximum BP (overshoot): {np.max(pressure[phase4_mask]):.1f} mmHg")
    
    print(f"\nValsalva Ratio: {valsalva_ratio:.2f}")
    print(f"Interpretation: ", end="")
    if valsalva_ratio >= 1.21:
        print("Normal autonomic function (ratio ≥1.21)")
    elif valsalva_ratio >= 1.10:
        print("Borderline autonomic function (1.10-1.20)")
    else:
        print("Abnormal - suggests autonomic neuropathy (ratio <1.10)")

print("\n" + "="*60)

## Part 2: Autonomic Dysfunction - Diabetic Neuropathy

Simulate Valsalva response in a patient with diabetic autonomic neuropathy (impaired baroreflex).

In [ ]:
from src.autonomic.baroreflex import BaroreflexParameters

# Create autonomic system with impaired baroreflex
impaired_baroreflex = BaroreflexParameters(
    max_firing_rate=50.0,      # Reduced from 100 (impaired baroreceptors)
    sigmoid_slope=0.05,        # Reduced from 0.1 (blunted response)
    vagal_gain=0.3,            # Reduced from 0.6 (decreased vagal)
    sympathetic_gain=0.3,      # Reduced from 0.6 (decreased sympathetic)
)

impaired_autonomic_params = AutonomicParameters(
    baseline_vagal_tone=0.4,   # Reduced from 0.7
    baseline_sympathetic_tone=0.4,  # Increased from 0.3
)

ans_impaired = AutonomicNervousSystem(
    params=impaired_autonomic_params,
    baroreflex_params=impaired_baroreflex
)

# Simulate Valsalva with impaired autonomic function
results_impaired = ans_impaired.simulate_valsalva_maneuver(
    duration=20.0,
    strain_duration=10.0,
    strain_pressure=40.0,
    dt=0.01
)

# Extract time series
times_imp = np.array(results_impaired['time'])
pressure_imp = np.array(results_impaired['pressure'])
hr_imp = np.array(results_impaired['heart_rate'])
phase_imp = np.array(results_impaired['phase'])

# Compare normal vs impaired
fig, axes = plt.subplots(2, 1, figsize=(14, 10), sharex=True)

# Phase annotations for both plots
for ax in axes:
    for i in range(1, 5):
        phase_mask = phase == i
        if np.any(phase_mask):
            t_start = times[phase_mask][0]
            t_end = times[phase_mask][-1]
            ax.axvspan(t_start, t_end, alpha=0.15, color=phase_colors[i-1])

# Plot 1: Blood Pressure Comparison
axes[0].plot(times, pressure, 'b-', linewidth=2.5, label='Normal', alpha=0.8)
axes[0].plot(times_imp, pressure_imp, 'r--', linewidth=2.5, label='Diabetic Neuropathy', alpha=0.8)
axes[0].axhline(93, color='gray', linestyle=':', alpha=0.5)
axes[0].set_ylabel('Mean Arterial Pressure (mmHg)', fontsize=11, fontweight='bold')
axes[0].set_title('Valsalva Maneuver - Normal vs Autonomic Neuropathy', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=11, loc='upper right')
axes[0].grid(True, alpha=0.3)

# Plot 2: Heart Rate Comparison
axes[1].plot(times, heart_rate, 'b-', linewidth=2.5, label='Normal', alpha=0.8)
axes[1].plot(times_imp, hr_imp, 'r--', linewidth=2.5, label='Diabetic Neuropathy', alpha=0.8)
axes[1].axhline(75, color='gray', linestyle=':', alpha=0.5)
axes[1].set_xlabel('Time (seconds)', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Heart Rate (bpm)', fontsize=11, fontweight='bold')
axes[1].legend(fontsize=11, loc='upper right')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Compute Valsalva ratios
phase2_mask_imp = phase_imp == 2
phase4_mask_imp = phase_imp == 4

if np.any(phase2_mask_imp) and np.any(phase4_mask_imp):
    hr_phase2_max_imp = np.max(hr_imp[phase2_mask_imp])
    hr_phase4_min_imp = np.min(hr_imp[phase4_mask_imp])
    valsalva_ratio_imp = hr_phase2_max_imp / hr_phase4_min_imp
    
    print("\n" + "="*60)
    print("COMPARISON: NORMAL vs DIABETIC NEUROPATHY")
    print("="*60)
    print(f"\nValsalva Ratio:")
    print(f"  Normal: {valsalva_ratio:.2f} (healthy response)")
    print(f"  Neuropathy: {valsalva_ratio_imp:.2f} (impaired)")
    
    print(f"\nPhase II HR Increase:")
    print(f"  Normal: +{hr_phase2_max - 75:.1f} bpm")
    print(f"  Neuropathy: +{hr_phase2_max_imp - 75:.1f} bpm (blunted)")
    
    print(f"\nPhase IV BP Overshoot:")
    print(f"  Normal: {np.max(pressure[phase4_mask]):.1f} mmHg")
    print(f"  Neuropathy: {np.max(pressure_imp[phase4_mask_imp]):.1f} mmHg (reduced)")
    
    print(f"\nClinical Interpretation:")
    print(f"  - Reduced Valsalva ratio indicates autonomic dysfunction")
    print(f"  - Blunted Phase II tachycardia suggests impaired baroreflex")
    print(f"  - Reduced Phase IV overshoot indicates poor vagal reactivation")
    print(f"  - Pattern consistent with diabetic autonomic neuropathy")
    print("\n" + "="*60)

## Part 3: Heart Failure - Square Wave Response

In advanced heart failure, the Valsalva response becomes a characteristic "square wave" pattern due to inability to compensate for reduced venous return.

In [ ]:
# Simulate heart failure with severely impaired cardiac reserve
# In HF, even with intact baroreceptors, the heart cannot increase output

hf_params = AutonomicParameters(
    baseline_vagal_tone=0.5,
    baseline_sympathetic_tone=0.6,  # Chronically elevated
    max_sympathetic_hr_effect=30.0,  # Reduced from 60 (impaired HR reserve)
    max_sympathetic_contractility=1.2,  # Reduced from 2.0 (impaired inotropy)
)

ans_hf = AutonomicNervousSystem(params=hf_params)

results_hf = ans_hf.simulate_valsalva_maneuver(
    duration=20.0,
    strain_duration=10.0,
    strain_pressure=40.0,
    dt=0.01
)

times_hf = np.array(results_hf['time'])
pressure_hf = np.array(results_hf['pressure'])
hr_hf = np.array(results_hf['heart_rate'])

# Plot all three conditions
fig, axes = plt.subplots(2, 1, figsize=(14, 10), sharex=True)

# Blood Pressure
axes[0].plot(times, pressure, 'g-', linewidth=2.5, label='Normal', alpha=0.8)
axes[0].plot(times_imp, pressure_imp, 'orange', linestyle='--', linewidth=2.5, 
            label='Diabetic Neuropathy', alpha=0.8)
axes[0].plot(times_hf, pressure_hf, 'r-', linewidth=2.5, label='Heart Failure', alpha=0.8)
axes[0].axhline(93, color='gray', linestyle=':', alpha=0.5)
axes[0].set_ylabel('Mean Arterial Pressure (mmHg)', fontsize=11, fontweight='bold')
axes[0].set_title('Valsalva Responses Across Pathological States', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)

# Heart Rate
axes[1].plot(times, heart_rate, 'g-', linewidth=2.5, label='Normal', alpha=0.8)
axes[1].plot(times_imp, hr_imp, 'orange', linestyle='--', linewidth=2.5, 
            label='Diabetic Neuropathy', alpha=0.8)
axes[1].plot(times_hf, hr_hf, 'r-', linewidth=2.5, label='Heart Failure', alpha=0.8)
axes[1].axhline(75, color='gray', linestyle=':', alpha=0.5)
axes[1].set_xlabel('Time (seconds)', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Heart Rate (bpm)', fontsize=11, fontweight='bold')
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3)

for ax in axes:
    for i in range(1, 5):
        phase_mask = phase == i
        if np.any(phase_mask):
            t_start = times[phase_mask][0]
            t_end = times[phase_mask][-1]
            ax.axvspan(t_start, t_end, alpha=0.1, color=phase_colors[i-1])

plt.tight_layout()
plt.show()

print("\n" + "="*60)
print("CLINICAL INTERPRETATION SUMMARY")
print("="*60)
print("\n1. NORMAL RESPONSE:")
print("   - Clear 4-phase pattern")
print("   - Vigorous Phase II tachycardia (baroreflex compensation)")
print("   - Prominent Phase IV overshoot and bradycardia")
print("   - Valsalva ratio ≥1.21")

print("\n2. DIABETIC AUTONOMIC NEUROPATHY:")
print("   - Preserved 4-phase structure but blunted responses")
print("   - Reduced Phase II HR increase (impaired baroreflex)")
print("   - Diminished Phase IV overshoot (poor vagal recovery)")
print("   - Valsalva ratio <1.21 (often <1.10 in severe cases)")

print("\n3. HEART FAILURE (SQUARE WAVE):")
print("   - Loss of distinct phases (square wave pattern)")
print("   - Minimal HR response despite intact autonomic nerves")
print("   - No Phase IV overshoot (limited cardiac reserve)")
print("   - Indicates NYHA Class III-IV heart failure")

print("\n" + "="*60)
print("DIAGNOSTIC VALUE")
print("="*60)
print("\nThe Valsalva maneuver is a powerful bedside test that:")
print("  • Assesses baroreflex sensitivity without equipment")
print("  • Differentiates autonomic vs cardiac dysfunction")
print("  • Predicts mortality post-MI (ATRAMI study)")
print("  • Guides therapy in heart failure and dysautonomia")
print("\nReference: Ewing DJ (1978) Br Heart J 40:163-169")
print("="*60)

## Summary and Clinical Pearls

### Key Takeaways

**1. Normal Valsalva Response:**
- Valsalva ratio ≥1.21 indicates intact autonomic function
- Phase II: Compensatory tachycardia (baroreflex-mediated)
- Phase IV: BP overshoot and reflex bradycardia

**2. Pathological Patterns:**
- **Autonomic Neuropathy**: Blunted but preserved 4-phase pattern, ratio <1.21
- **Heart Failure**: Square wave pattern, minimal HR response
- **Hypovolemia**: Exaggerated Phase II hypotension, prolonged recovery

**3. Clinical Applications:**
- Bedside autonomic testing (no equipment needed)
- Post-MI risk stratification
- Heart failure severity assessment
- Syncope evaluation
- Diabetic neuropathy screening

**4. Contraindications:**
- Proliferative retinopathy (risk of vitreous hemorrhage)
- Recent MI or stroke (<3 months)
- Severe aortic stenosis
- Acute glaucoma

### References

- Goldberger et al. (2019) AHA Scientific Statement on Autonomic Testing
- Ewing DJ (1978) Practical bedside investigation of diabetic autonomic failure
- Freeman R (2006) Assessment of cardiovascular autonomic function
- La Rovere et al. (1998) Baroreflex sensitivity and heart-rate variability in prediction of mortality

---
© 2025 Multi-Heart-Model Project | MIT License